In [1]:
import pandas as pd

# ==========================================
# 1. 提取当前订单簿截面数据 (Order Book Snapshot)
# ==========================================
# 格式: {Price: Volume}
others_bids = {30: 30000, 29: 5000, 28: 12000, 27: 28000}
asks = {28: 40000, 31: 20000, 32: 20000, 33: 30000}

# 资产参数设定 (Dryland Flax)
TRUE_VALUE = 30  # 官方回收价
FEE = 0          # 手续费

def simulate_auction(my_price, my_qty):
    """
    模拟交易所单一价格结算引擎，返回：(预期利润, 结算价, 你的实际成交量)
    """
    # 注入我们的订单
    all_bids = dict(others_bids)
    all_bids[my_price] = all_bids.get(my_price, 0) + my_qty

    # 获取所有潜在的价格档位
    all_prices = sorted(set(all_bids.keys()).union(set(asks.keys())))

    max_vol = -1
    clearing_price = -1

    # --- 阶段 A: 寻找结算价 (Clearing Price) ---
    for p in all_prices:
        demand = sum(v for price, v in all_bids.items() if price >= p)
        supply = sum(v for price, v in asks.items() if price <= p)
        vol = min(demand, supply)

        if vol > max_vol:
            max_vol = vol
            clearing_price = p
        elif vol == max_vol:
            # 交易所规则：平局时选择较高的价格
            if p > clearing_price:
                clearing_price = p

    # --- 阶段 B: 计算你的真实成交量 (Fill Logic) ---
    supply_at_cp = sum(v for price, v in asks.items() if price <= clearing_price)
    others_demand_gt_cp = sum(v for price, v in others_bids.items() if price > clearing_price)

    if my_price > clearing_price:
        # 价格优先：利用高价插队，只要总供给够，你必然全额成交
        my_fill = my_qty
    elif my_price == clearing_price:
        # 时间劣势：因为你是最后提交的，同价位其他人的单子优先吃掉流动性
        others_demand_eq_cp = sum(v for price, v in others_bids.items() if price == clearing_price)
        supply_left_for_me = supply_at_cp - others_demand_gt_cp - others_demand_eq_cp
        my_fill = min(my_qty, max(0, supply_left_for_me))
    else:
        # 报价过低，无法成交
        my_fill = 0

    # --- 阶段 C: 结算利润 ---
    cost = clearing_price + FEE
    profit = (TRUE_VALUE - cost) * my_fill

    return profit, clearing_price, my_fill

# ==========================================
# 2. 策略求解 (Grid Search)
# ==========================================
best_profit = -1
best_strategy = None

# 我们只需要测试有意义的出价区间 (28, 29, 30) 和潜在的数量区间
for test_price in [28, 29, 30]:
    for test_qty in range(0, 10000):  # 探索到10k足够覆盖边界
        profit, cp, fill = simulate_auction(test_price, test_qty)
        
        if profit > best_profit:
            best_profit = profit
            best_strategy = (test_price, test_qty, cp, fill)

# ==========================================
# 3. 输出执行结果
# ==========================================
print("=== Dryland Flax 最优做市套利策略 ===")
print(f"你应该提交的 报价 (Bid Price): {best_strategy[0]}")
print(f"你应该提交的 数量 (Quantity) : {best_strategy[1]}")
print("-----------------------------------")
print(f"引擎预估结算价 (Clearing Price): {best_strategy[2]}")
print(f"你的实际成交量 (Your Fill)     : {best_strategy[3]}")
print(f"最大化无风险利润 (Max Profit)  : {best_profit}")

=== Dryland Flax 最优做市套利策略 ===
你应该提交的 报价 (Bid Price): 30
你应该提交的 数量 (Quantity) : 9999
-----------------------------------
引擎预估结算价 (Clearing Price): 29
你的实际成交量 (Your Fill)     : 9999
最大化无风险利润 (Max Profit)  : 9999


In [2]:
from math import ceil

def choose_clearing_price(bids, asks):
    """
    bids / asks: dict[price] = qty
    返回: (clearing_price, traded_volume)
    """
    prices = sorted(set(bids) | set(asks))
    best_price = None
    best_vol = -1

    for p in prices:
        buy_qty = sum(q for px, q in bids.items() if px >= p)
        sell_qty = sum(q for px, q in asks.items() if px <= p)
        traded = min(buy_qty, sell_qty)

        if traded > best_vol or (traded == best_vol and (best_price is None or p > best_price)):
            best_vol = traded
            best_price = p

    return best_price, best_vol


def my_filled_qty(base_bids, asks, my_price, my_qty):
    """
    在 base_bids 上加上我的买单后，计算：
    - clearing price
    - total traded volume
    - 我的真实成交量
    """
    bids = dict(base_bids)
    bids[my_price] = bids.get(my_price, 0) + my_qty

    clearing_price, traded_volume = choose_clearing_price(bids, asks)

    # 按买方价格优先、时间优先分配成交
    remaining = traded_volume
    my_fill = 0

    for px in sorted(bids.keys(), reverse=True):
        if px < clearing_price or remaining <= 0:
            continue

        total_here = bids[px]
        old_here = base_bids.get(px, 0)
        mine_here = my_qty if px == my_price else 0

        exec_here = min(remaining, total_here)

        # 同价位旧单先于我成交
        mine_exec = max(0, min(mine_here, exec_here - old_here))
        my_fill += mine_exec
        remaining -= exec_here

    return clearing_price, traded_volume, my_fill


def profit_for_order(base_bids, asks, my_price, my_qty, resale_price, fee_per_unit=0.0):
    """
    fee_per_unit: 总费用（买入+卖出）按每单位合计
    例如 EMBER_MUSHROOM 是 0.10
    """
    cp, tv, my_fill = my_filled_qty(base_bids, asks, my_price, my_qty)
    unit_profit = resale_price - fee_per_unit - cp
    total_profit = my_fill * unit_profit
    return {
        "bid_price": my_price,
        "bid_qty": my_qty,
        "clearing_price": cp,
        "traded_volume": tv,
        "my_fill": my_fill,
        "unit_profit": unit_profit,
        "total_profit": total_profit,
    }


def solve_optimal(base_bids, asks, resale_price, fee_per_unit=0.0, qty_step=1):
    """
    穷举最优买单。
    qty_step=1   -> 精确到 1 单位
    qty_step=1000 -> 只搜 1000 的整数倍
    """
    # 价格只要搜到 max(最高现有买价+1, 最高现有卖价) 就够了
    max_bid = max(base_bids) if base_bids else 0
    max_ask = max(asks) if asks else 0
    min_price = min(list(base_bids.keys()) + list(asks.keys()))
    max_price = max(max_bid + 1, max_ask)

    # 数量上限：搜到所有可见卖盘总量即可
    max_qty = sum(asks.values())

    best = None

    for p in range(min_price, max_price + 1):
        for q in range(0, max_qty + 1, qty_step):
            res = profit_for_order(base_bids, asks, p, q, resale_price, fee_per_unit)

            if best is None:
                best = res
            else:
                # 先比利润，再偏好更低买价，再偏好更小数量
                # 只是为了在多个等价最优解里返回一个更“克制”的解
                if (
                    res["total_profit"] > best["total_profit"]
                    or (
                        res["total_profit"] == best["total_profit"]
                        and (
                            res["bid_price"] < best["bid_price"]
                            or (
                                res["bid_price"] == best["bid_price"]
                                and res["bid_qty"] < best["bid_qty"]
                            )
                        )
                    )
                ):
                    best = res

    return best


if __name__ == "__main__":
    # -------- DRYLAND_FLAX --------
    flax_bids = {
        30: 30000,
        29: 5000,
        28: 12000,
        27: 28000,
    }
    flax_asks = {
        28: 40000,
        31: 20000,
        32: 20000,
        33: 30000,
    }

    # -------- EMBER_MUSHROOM --------
    mushroom_bids = {
        20: 43000,
        19: 17000,
        18: 6000,
        17: 5000,
        16: 10000,
        15: 5000,
        14: 10000,
        13: 7000,
    }
    mushroom_asks = {
        12: 20000,
        13: 25000,
        14: 35000,
        15: 6000,
        16: 5000,
        17: 0,
        18: 10000,
        19: 12000,
    }

    print("=== 精确到 1 单位 ===")
    best_flax = solve_optimal(flax_bids, flax_asks, resale_price=30.0, fee_per_unit=0.0, qty_step=1)
    best_mush = solve_optimal(mushroom_bids, mushroom_asks, resale_price=20.0, fee_per_unit=0.10, qty_step=1)
    print("DRYLAND_FLAX:", best_flax)
    print("EMBER_MUSHROOM:", best_mush)

    print("\n=== 只搜 1000 的整数倍 ===")
    best_flax_k = solve_optimal(flax_bids, flax_asks, resale_price=30.0, fee_per_unit=0.0, qty_step=1000)
    best_mush_k = solve_optimal(mushroom_bids, mushroom_asks, resale_price=20.0, fee_per_unit=0.10, qty_step=1000)
    print("DRYLAND_FLAX (1000-step):", best_flax_k)
    print("EMBER_MUSHROOM (1000-step):", best_mush_k)

=== 精确到 1 单位 ===
DRYLAND_FLAX: {'bid_price': 30, 'bid_qty': 9999, 'clearing_price': 29, 'traded_volume': 40000, 'my_fill': 9999, 'unit_profit': 1.0, 'total_profit': 9999.0}
EMBER_MUSHROOM: {'bid_price': 17, 'bid_qty': 19999, 'clearing_price': 16, 'traded_volume': 91000, 'my_fill': 19999, 'unit_profit': 3.8999999999999986, 'total_profit': 77996.09999999998}

=== 只搜 1000 的整数倍 ===
DRYLAND_FLAX (1000-step): {'bid_price': 30, 'bid_qty': 9000, 'clearing_price': 29, 'traded_volume': 40000, 'my_fill': 9000, 'unit_profit': 1.0, 'total_profit': 9000.0}
EMBER_MUSHROOM (1000-step): {'bid_price': 19, 'bid_qty': 40000, 'clearing_price': 18, 'traded_volume': 101000, 'my_fill': 40000, 'unit_profit': 1.8999999999999986, 'total_profit': 75999.99999999994}
